<a href="https://colab.research.google.com/github/mythien107/busad878/blob/main/function_calling_demo_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Function Calling in Practice — Gemini Demo

This notebook walks through a complete function calling interaction:

1. **Define** — Tell the model what tools exist
2. **Decide** — The model chooses whether and which tool to call
3. **Execute** — Your system runs the actual function
4. **Return** — Send results back to the model
5. **Respond** — The model incorporates real data into its answer

Run each cell in order and watch the magic happen!

---
## Step 0: Setup
Install the Gemini SDK and configure your API key.

In [ ]:
# Uninstall potentially conflicting packages
!pip uninstall -y google-generativeai google-genai
# Install the recommended package
!pip install -q google-genai

print("Packages updated!")

In [ ]:
# Configure your API key
# Option 1: Use Colab's secrets manager (recommended)
# Option 2: Paste your key directly (less secure)

import google.genai as genai

# Try to get key from Colab secrets first
try:
    from google.colab import userdata
    API_KEY = userdata.get('GOOGLE_API_KEY')
    print("API key loaded from Colab secrets")
except:
    # Fall back to manual entry
    API_KEY = input("Enter your Google AI API key: ")
    print("API key set manually")

---
## Step 1: Create Our "Database"

In a real application, this would be your actual inventory system, CRM, or any external data source. For this demo, we'll use a simple pandas DataFrame.

In [ ]:
import pandas as pd
from io import StringIO

# Our "database" - in reality this would be PostgreSQL, an API, etc.
INVENTORY_CSV = """product_id,product_name,quantity,warehouse,reorder_point,last_updated
SKU-12345,Widget Pro,1523,ALL,2000,2027-01-20T10:30:00Z
SKU-12345,Widget Pro,892,DAL,2000,2027-01-20T09:15:00Z
SKU-12345,Widget Pro,631,CHI,2000,2027-01-20T10:30:00Z
SKU-67890,Gadget Plus,5420,ALL,1000,2027-01-19T14:00:00Z
SKU-67890,Gadget Plus,3200,DAL,1000,2027-01-19T14:00:00Z
SKU-67890,Gadget Plus,2220,CHI,1000,2027-01-19T12:30:00Z
SKU-11111,Basic Unit,187,ALL,500,2027-01-20T08:45:00Z
SKU-11111,Basic Unit,187,CHI,500,2027-01-20T08:45:00Z
SKU-99999,Premium Pack,0,ALL,100,2027-01-18T16:30:00Z
SKU-99999,Premium Pack,0,DAL,100,2027-01-18T16:30:00Z"""

inventory_df = pd.read_csv(StringIO(INVENTORY_CSV))

# Show the inventory summary
print("INVENTORY DATABASE")
print("=" * 60)
display(inventory_df[inventory_df["warehouse"] == "ALL"][["product_id", "product_name", "quantity", "reorder_point"]])

---
## Step 2: Define the Tools

This is where we tell the model what tools are available. Notice:
- **Clear descriptions** — The model uses these to decide *when* to call each tool
- **Typed parameters** — The model extracts these from natural language
- **Required vs optional** — `product_id` is required, `warehouse` is optional

In [ ]:
# Define tools using Gemini's format
tools = [
    {
        "function_declarations": [
            {
                "name": "get_inventory",
                "description": """Get current inventory level for a product.
Returns quantity and warehouse location.
Use when the user asks about stock levels or availability.""",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "product_id": {
                            "type": "string",
                            "description": "The product SKU (e.g., 'SKU-12345')"
                        },
                        "warehouse": {
                            "type": "string",
                            "description": "Warehouse code (e.g., 'DAL', 'CHI'). Omit to get total across all warehouses."
                        }
                    },
                    "required": ["product_id"]
                }
            },
            {
                "name": "calculator",
                "description": """Perform mathematical calculations.
Use for any arithmetic or numerical comparison to ensure accuracy.
Accepts standard math expressions.""",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "expression": {
                            "type": "string",
                            "description": "Mathematical expression (e.g., '1523 < 2000', '2000 - 1523')"
                        }
                    },
                    "required": ["expression"]
                }
            }
        ]
    }
]

print("TOOLS DEFINED")
print("=" * 60)
print("")
print("1--get_inventory")
print("    → Retrieves stock levels from our database")
print("    → Parameters: product_id (required), warehouse (optional)")
print("")
print("2--calculator")
print("    → Performs accurate arithmetic")
print("    → Parameters: expression (required)")
print("")
print("Tools ready!")

---
## Step 3: Implement the Tool Functions

**This is YOUR code** — the model never runs this directly. It just *requests* that you run it. This is the key security boundary in function calling.

In [ ]:
def get_inventory(product_id: str, warehouse: str = None) -> dict:
    """
    Query our inventory database.
    In production: This would call your real database/API
    """
    print(f"     Querying database for {product_id}...")

    if warehouse:
        filtered = inventory_df[
            (inventory_df["product_id"] == product_id) &
            (inventory_df["warehouse"] == warehouse)
        ]
    else:
        filtered = inventory_df[
            (inventory_df["product_id"] == product_id) &
            (inventory_df["warehouse"] == "ALL")
        ]

    if len(filtered) == 0:
        return {"error": f"Product {product_id} not found"}

    row = filtered.iloc[0]
    result = {
        "product_id": row["product_id"],
        "product_name": row["product_name"],
        "quantity": int(row["quantity"]),
        "warehouse": warehouse or "ALL",
        "reorder_point": int(row["reorder_point"]),
        "last_updated": row["last_updated"]
    }
    print(f"     Found: {result['quantity']} units of {result['product_name']}")
    return result


def calculator(expression: str) -> dict:
    """
    Evaluate a mathematical expression.
    In production: Use a safe math parser library
    """
    print(f"     Calculating: {expression}")
    try:
        result = eval(expression)  # Note: Use a safe parser in production!
        print(f"     Result: {result}")
        return {"expression": expression, "result": result}
    except Exception as e:
        return {"error": str(e)}


def execute_tool(function_name: str, arguments: dict) -> dict:
    """Route tool calls to the appropriate function"""
    if function_name == "get_inventory":
        return get_inventory(
            product_id=arguments.get("product_id"),
            warehouse=arguments.get("warehouse")
        )
    elif function_name == "calculator":
        return calculator(expression=arguments.get("expression"))
    else:
        return {"error": f"Unknown function: {function_name}"}


print(" TOOL FUNCTIONS IMPLEMENTED")
print("=" * 60)
print("")
print("These functions are YOUR code. The model will request")
print("that you run them, but it never executes them directly.")
print("")
print(" Functions ready!")

---
## Step 4: Initialize the Model

We create a Gemini model instance with our tools attached, then start a chat session.

In [ ]:
# Initialize the Gemini Client
client = genai.Client(api_key=API_KEY)

# Use client.models.list() instead of list_models()
for model in client.models.list():
  if 'gemini' in model.name:
    print(model.name)

In [ ]:
# Start a chat session and pass the tools in the config
chat = client.chats.create(
    model="gemini-2.5-flash-lite",
    config={"tools": tools}
)

print(" MODEL INITIALIZED")
print("=" * 60)
print("")
print("Model: gemini-2.5-flash-lite")
print("Tools: get_inventory, calculator")
print("")
print(" Ready for conversation!")

---
##  Step 5: The Demo!

Now let's see function calling in action. We'll send a user question that requires:
1. Fetching real inventory data
2. Comparing it to a threshold

Watch how the model decides to use tools!

In [ ]:
# The user's question
user_query = "What's the inventory for SKU-12345, and is it below our reorder point of 2,000 units?"

print(" USER QUERY")
print("=" * 60)
print(f"\"{user_query}\"")
print("")
print(" Sending to Gemini...")

In [ ]:
# Send the message and get initial response
response = chat.send_message(user_query)

print(" INITIAL RESPONSE RECEIVED")
print("=" * 60)
print("")

# Check what the model returned
for part in response.candidates[0].content.parts:
    if hasattr(part, "function_call") and part.function_call.name:
        print(" MODEL WANTS TO USE A TOOL!")
        print(f"   Function: {part.function_call.name}")
        print(f"   Arguments: {dict(part.function_call.args)}")
        print("")
    elif hasattr(part, "text") and part.text:
        print(f" Text: {part.text}")

###  What Just Happened?

The model **did not** make up an inventory number. Instead, it:
1. Recognized it needs real data
2. Chose the correct tool (`get_inventory`)
3. Extracted the product ID from natural language
4. Returned a structured request for us to execute

Now let's execute the tool and send the result back!

In [ ]:
import concurrent.futures

# Process tool calls in a loop (model might need multiple tools)
iteration = 0

while True:
    # The new SDK has a convenient .function_calls property
    function_calls = response.function_calls

    # If no function calls, we have the final response
    if not function_calls:
        break

    iteration += 1
    print(f"\n TOOL EXECUTION #{iteration}")
    print("=" * 60)

    def process_call(function_call):
        fn_name = function_call.name
        fn_args = dict(function_call.args)

        print(f"\n Function requested: {fn_name}")
        print(f" Arguments: {fn_args}")
        print(f"  Executing...")

        # YOUR CODE RUNS HERE - Execute the tool
        tool_result = execute_tool(fn_name, fn_args)
        print(f" Result: {tool_result}")

        return genai.types.Part.from_function_response(
            name=fn_name,
            response={"result": tool_result}
        )

    # Execute tools concurrently
    with concurrent.futures.ThreadPoolExecutor() as executor:
        function_responses = list(executor.map(process_call, function_calls))

    print(f"\n Sending results back to model...")
    # Send all results back to the model at once
    response = chat.send_message(function_responses)

print("\n All tool calls complete!")

In [ ]:
# Display the final response
print(" FINAL RESPONSE")
print("=" * 60)
print("")

print(response.text)

---
##  What Just Happened?

Let's recap the complete cycle:

| Step | What Happened |
|------|---------------|
| 1. **Define** | We told Gemini about `get_inventory` and `calculator` |
| 2. **User Query** | "What's the inventory for SKU-12345..." |
| 3. **Decide** | Model chose `get_inventory` (not `calculator` first!) |
| 4. **Execute** | OUR code queried the database |
| 5. **Return** | We sent `{quantity: 1523, ...}` back |
| 6. **Respond** | Model gave a complete answer with real data |

### Key Insights:
- The model **never touched our database** — it only requested data
- It **extracted parameters** from natural language automatically
- It **reasoned about tool order** — inventory first, then comparison
- It performed simple math internally (1523 < 2000) without the calculator

---
##  Try It Yourself!

Run the cell below to ask your own questions. Try:
- "What's in stock for SKU-67890?"
- "Compare inventory levels for SKU-11111 and SKU-99999"
- "Which products are below their reorder point?"
- "How many Widget Pros are in the Dallas warehouse?"

In [ ]:
import concurrent.futures

# Interactive cell - try your own queries!
print("Type 'quit' or 'exit' to stop.\n")

while True:
    your_query = input("Ask a question about inventory: ")

    if your_query.lower() in ['quit', 'exit', 'q']:
        print("\nEnding conversation.")
        break

    print("\n Your question:", your_query, "\n")

    # Send to model
    response = chat.send_message(your_query)

    # Handle any tool calls
    while True:
        function_calls = response.function_calls
        if not function_calls:
            break

        def process_call(function_call):
            fn_name = function_call.name
            fn_args = dict(function_call.args)

            print(f" Tool call: {fn_name}({fn_args})")
            tool_result = execute_tool(fn_name, fn_args)
            print(f" Result: {tool_result}\n")

            return genai.types.Part.from_function_response(
                name=fn_name,
                response={"result": tool_result}
            )

        # Execute tools concurrently
        with concurrent.futures.ThreadPoolExecutor() as executor:
            function_responses = list(executor.map(process_call, function_calls))

        response = chat.send_message(function_responses)

    # Print final response
    print(" Response:")
    print(response.text)
    print("-" * 60 + "\n")

---
##  Summary

Function calling follows the same pattern across all providers:

```
┌─────────────┐     ┌─────────────┐     ┌─────────────┐
│   Define    │ ──▶│   Decide    │ ──▶│   Execute   │
│   Tools     │     │  (Model)    │     │ (Your Code) │
└─────────────┘     └─────────────┘     └──────┬──────┘
                                               │
                    ┌─────────────┐     ┌──────▼──────┐
                    │   Respond   │ ◀──│   Return    │
                    │  (Model)    │     │   Result    │
                    └─────────────┘     └─────────────┘
```

The model is the **brain** that decides what to do.
Your code is the **hands** that actually do it.

This separation is what makes function calling both powerful and safe!